Multiple Linear Regression:

This notebook walks through Multiple Linear Regression end-to-end:

1. Load / generate data
2. Explore the data (shape, preview, 3D visualization)
3. Train a model using scikit-learn's `LinearRegression`
4. Evaluate the model with MAE, MSE, and R² metrics
5. Implement our own Multiple Linear Regression class using the Normal Equation
6. Visualize and compare the from-scratch model against scikit-learn's model


1. Import Libraries

In [1]:
# Importing Libraries
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

2. Load / Generate Data

We generate a synthetic regression dataset with 2 informative features so that we can
visualize the regression plane in 3D later.

In [2]:
X, y = make_regression(
    n_samples=100,
    n_features=2, #input features
    n_informative=2, # it means that both the features affect the output rather than one feature
    n_targets=1, # output
    noise=50,
    random_state=42  # ensures reproducible dataset across runs
)

3. Build a DataFrame

Wrapping the raw numpy arrays in a `DataFrame` makes exploration and plotting easier.

In [4]:
df = pd.DataFrame({'feature1': X[:, 0], 'feature2': X[:, 1], 'target': y})

3.1 Inspect the data

In [5]:
df.shape

(100, 3)

In [6]:
df.head()

,feature1,feature2,target
0,-1.191303,0.656554,-22.779796
1,0.058209,-1.142970,-107.569629
2,0.586857,2.190456,201.122932
3,0.473238,-0.072829,1.480178
4,0.738467,0.171368,111.798503


In [7]:
fig = px.scatter_3d(df, x='feature1', y='feature2', z='target', title='Raw Data: feature1 vs feature2 vs target')
fig.show()

5. Train / Test Split

We hold out 20% of the data to evaluate how well the model generalizes to unseen data.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

6. Train a Model with Scikit-Learn

We start with scikit-learn's built-in `LinearRegression` as our baseline / reference model.

In [9]:
sklearn_lr = LinearRegression()
sklearn_lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[81.06,72.39]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.364
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](2,)","[9.17,7.63]"


In [10]:
y_pred_sklearn = sklearn_lr.predict(X_test)
y_pred_sklearn

array([ -55.14997532,  -94.41780237, -152.08501951,   23.8116796 ,
         48.26529266,  -46.51097392,   44.55563019,  -69.09281689,
         13.4108607 , -172.94096688,    3.64775442,  152.43304515,
        -80.49588243, -115.92262562, -139.8139105 ,   50.74885   ,
         87.67488507,  111.64539599, -115.36436666,  -17.80277998])

7. Evaluate the Scikit-Learn Model

We use three standard regression metrics:

- MAE (Mean Absolute Error): average absolute difference between prediction and truth.
- MSE (Mean Squared Error): penalizes larger errors more heavily than MAE.
- R² score: proportion of variance in the target explained by the model (1.0 = perfect).

In [11]:
print("MAE is ", mean_absolute_error(y_test, y_pred_sklearn))
print("MSE is ", mean_squared_error(y_test, y_pred_sklearn))
print("R2 score is ", r2_score(y_test, y_pred_sklearn))

MAE is  47.0876578106196
MSE is  3639.6081061143377
R2 score is  0.655121125764055


8. Visualize the Best-Fit Plane

With 2 features, the "line of best fit" from simple linear regression becomes a
plane in 3D. We build a grid over the feature space, predict the target at every
grid point, and render it as a surface alongside the actual data points.

In [19]:
import plotly.graph_objects as go

# 1. Ensure x and y match your actual data range
x_range = np.linspace(df['feature1'].min(), df['feature1'].max(), 10)
y_range = np.linspace(df['feature2'].min(), df['feature2'].max(), 10)
xGrid, yGrid = np.meshgrid(x_range, y_range)

# 2. Predict the surface height
final = np.c_[xGrid.ravel(), yGrid.ravel()]
z_final = sklearn_lr.predict(final).reshape(10, 10)

# 3. Build the plot from scratch using ONLY go (no px)
fig = go.Figure()

# Add the Scatter Dots
fig.add_trace(go.Scatter3d(
    x=df['feature1'],
    y=df['feature2'],
    z=df['target'],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.8),
    name='Actual Data'
))

# Add the Best Fit Plane
fig.add_trace(go.Surface(
    x=x_range,
    y=y_range,
    z=z_final,
    colorscale='Reds',
    opacity=0.6,
    name='Best Fit Plane'
))

# Force the layout to show everything
fig.update_layout(
    title='Multiple Linear Regression: Best Fit Plane',
    scene=dict(
        xaxis_title='Feature 1',
        yaxis_title='Feature 2',
        zaxis_title='Target'
    ),
    width=800,
    height=800
)

fig.show()

9. Inspect the Learned Coefficients

`coef_` holds the learned weight for each feature, and `intercept_` is the bias term (β₀).
We'll hang on to these so we can compare them against our own implementation later.

In [20]:
sklearn_lr.coef_

array([81.05675748, 72.39437821])

In [21]:
sklearn_lr.intercept_

np.float64(5.363809072090564)

10. Implement Our Own Multiple Linear Regression Class

Now that we have a working baseline, let's implement Multiple Linear Regression
from scratch using the closed-form Normal Equation:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

where a column of 1's is prepended to `X` so the first coefficient (`betas[0]`)
represents the intercept, and the rest (`betas[1:]`) are the feature coefficients.

In [22]:
class MultipleLinearRegression:

    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # Prepend a column of 1's to X so we can solve for the intercept
        # (beta_0) in the same matrix equation as the feature coefficients.
        X_train = np.insert(X_train, 0, 1, axis=1)

        # Normal Equation: betas = (X^T X)^-1 X^T y
        betas = np.linalg.inv(X_train.T.dot(X_train)).dot(X_train.T).dot(y_train)

        self.intercept_ = betas[0]
        self.coef_ = betas[1:]

    def predict(self, X_test):
        # y_pred = X . coefficients + intercept
        y_pred = np.dot(X_test, self.coef_) + self.intercept_
        return y_pred

11. Train Our Custom Model

Same `X_train` / `y_train` split as before, so the comparison against scikit-learn
is apples-to-apples.

In [23]:
mul_lr = MultipleLinearRegression()
mul_lr.fit(X_train, y_train)

11.1 Sanity Check: Shapes

Quick check that the intercept column was added correctly, `X_train` should gain
one extra column (100% for the bias term) after `np.insert`.

In [24]:
X_train.shape

(80, 2)

In [25]:
np.insert(X_train, 0, 1, axis=1).shape

(80, 3)

12. Predict and Evaluate Our Custom Model

In [26]:
y_pred_custom = mul_lr.predict(X_test)
y_pred_custom

array([ -55.14997532,  -94.41780237, -152.08501951,   23.8116796 ,
         48.26529266,  -46.51097392,   44.55563019,  -69.09281689,
         13.4108607 , -172.94096688,    3.64775442,  152.43304515,
        -80.49588243, -115.92262562, -139.8139105 ,   50.74885   ,
         87.67488507,  111.64539599, -115.36436666,  -17.80277998])

In [27]:
print("MAE is ", mean_absolute_error(y_test, y_pred_custom))
print("MSE is ", mean_squared_error(y_test, y_pred_custom))
print("R2 score is ", r2_score(y_test, y_pred_custom))

MAE is  47.08765781061961
MSE is  3639.6081061143377
R2 score is  0.655121125764055


13. Compare Coefficients: Scikit-Learn vs. Our Implementation

In [28]:
print("scikit-learn coefficients:", sklearn_lr.coef_)
print("our model's coefficients:    ", mul_lr.coef_)
print()
print("scikit-learn intercept:", sklearn_lr.intercept_)
print("our model's intercept: ", mul_lr.intercept_)

print()
print("Coefficients match:", np.allclose(sklearn_lr.coef_, mul_lr.coef_))
print("Intercepts match:  ", np.isclose(sklearn_lr.intercept_, mul_lr.intercept_))

scikit-learn coefficients: [81.05675748 72.39437821]
our model's coefficients:     [81.05675748 72.39437821]

scikit-learn intercept: 5.363809072090564
our model's intercept:  5.363809072090572

Coefficients match: True
Intercepts match:   True
